# CE_TM_03 (Crossing Frequency)

## Libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import dash
from dash import dcc, html, Input, Output, State
import dash_bootstrap_components as dbc
import plotly.express as px
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go


## Import Files

In [2]:
path_to_file = "https://raw.githubusercontent.com/cesarlarasantana/VDS_2526_G04_Football/refs/heads/main/Datasets"

df_country = pd.read_csv(path_to_file+"/Country.csv")
df_match_goals = pd.read_csv(path_to_file+"/Match_Goals.csv")
df_match_shots_on = pd.read_csv(path_to_file+"/Match_Shots_On.csv")
df_match_fouls = pd.read_csv(path_to_file+"/Match_Fouls_Committed.csv")
df_match_cards = pd.read_csv(path_to_file+"/Match_Cards.csv")
df_team = pd.read_csv(path_to_file+"/Team.csv")
df_match = pd.read_csv(path_to_file+"/Match.csv")
df_cross = pd.read_csv(path_to_file+"/Match_Cross.csv")
df_player = pd.read_csv(path_to_file+"/Player.csv")
df_player_att = pd.read_csv(path_to_file+"/Player_Attributes.csv")
df_shots_on = pd.read_csv(path_to_file+"/Match_Shots_On.csv")
df_shots_off = pd.read_csv(path_to_file+"/Match_Shots_Off.csv")
df_position_ref = pd.read_csv(path_to_file+"/PositionReference.csv")

C:\Users\pci\AppData\Local\Temp\ipykernel_17008\3948160842.py:6: DtypeWarning: Columns (0: player1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_match_fouls = pd.read_csv(path_to_file+"/Match_Fouls_Committed.csv")


## Data Preparation

In [3]:
## Match results of teams ##
# Results of home games
home_results = df_match[["id", "season", "home_team_api_id", "home_team_goal", "away_team_goal"]].copy()
home_results = home_results.rename(columns={"id": "match_id","home_team_api_id": "team_api_id"})
home_results["result_group"] = np.where(home_results["home_team_goal"] > home_results["away_team_goal"], "Wins", "Draws and Losses")
# Results of away games
away_results = df_match[["id", "season", "away_team_api_id", "home_team_goal", "away_team_goal"]].copy()
away_results = away_results.rename(columns={"id": "match_id", "away_team_api_id": "team_api_id"})
away_results["result_group"] = np.where(away_results["away_team_goal"] > away_results["home_team_goal"], "Wins", "Draws and Losses")
# Create complete dataset (home and away games)
df_team_match_result = pd.concat([home_results, away_results], ignore_index=True)
df_team_match_result = df_team_match_result[["match_id", "season", "team_api_id", "result_group"]]

## Create time slots ##
time_slots = [0, 15, 30, 45, 60, 75, 90]
time_label = ["0-15", "15-30", "30-45", "45-60", "60-75", "75-90"]
time_dict = {"0-15": 1, "15-30": 2, "30-45": 3, "45-60": 4, "60-75": 5, "75-90": 6}

def add_time_slots(df):
    df = df.copy()
    df["elapsed"] = df["elapsed"].clip(lower=0, upper=90)
    df["time_slots"] = pd.cut(df["elapsed"],
                              bins=time_slots,
                              labels=time_label,
                              include_lowest=True,
                              right=True)
    df["time_label"] = df["time_slots"].astype(str)
    return df

## Crosses ##
df_just_crosses = df_cross.copy()
if "type" in df_just_crosses.columns:
    df_just_crosses = df_just_crosses[df_just_crosses["type"] == "cross"].copy()
df_just_crosses = add_time_slots(df_just_crosses)
df_just_crosses = (df_just_crosses
                   .groupby(["match_id", "team", "time_label"], observed=False)
                   .size()
                   .reset_index(name="crosses")
                   .rename(columns={"team": "team_api_id"}))


## Total shots ##
# Calculate total shots as shots on + shots off
df_total_shots = pd.concat([df_shots_on, df_shots_off], ignore_index=True)

df_total_shots = add_time_slots(df_total_shots)

df_total_shots = (df_total_shots
                  .groupby(["match_id", "team", "time_label"], observed=False)
                  .size()
                  .reset_index(name="total_shots")
                  .rename(columns={"team": "team_api_id"}))

## Create dataframe that devides every match into the times slots ##
# e.g.
# match | team     | time interval | shots | crosses |
# 1     | Dortmund | 1-15          | 0     | 0       |

# match + team + time interval
df_match_by_timeslot = df_team_match_result.copy()
df_time_slot = pd.DataFrame({"time_label": time_label})
df_base = df_match_by_timeslot.merge(df_time_slot, how="cross")

# match + team + time interval + shots + crosses
df_crossing = df_base.merge(df_total_shots, on=["match_id", "team_api_id", "time_label"], how="left")
df_crossing = df_crossing.merge(df_just_crosses, on=["match_id", "team_api_id", "time_label"], how="left")
df_crossing["total_shots"] = df_crossing["total_shots"].fillna(0)
df_crossing["crosses"] = df_crossing["crosses"].fillna(0)
df_crossing = df_crossing.merge(df_team[["team_api_id", "team_long_name"]], on="team_api_id", how="left")

## Aggregate dataframe and calculate cross shot percentage ##
df_crossing_agg = (df_crossing
                        .groupby(["team_api_id", "team_long_name", "result_group", "time_label"], observed=False)[["crosses", "total_shots"]]
                        .sum()
                        .reset_index())
# Calculate percentage of cross shots as crosses/(crosses + totalshots) * 100
df_crossing_agg["cross_shot_percentage"] = (df_crossing_agg["crosses"] /
                                                 (df_crossing_agg["crosses"] + df_crossing_agg["total_shots"]) * 100)
df_crossing_agg["cross_shot_percentage"] = (df_crossing_agg["cross_shot_percentage"].fillna(0))

## Team ranking for dropdown ##
# Goals: Only counting valid goals (n and p)
df_goals = df_match_goals[df_match_goals["goal_type"].isin(["n", "p"])].copy()
df_goals = df_goals.merge(df_match[["id", "season"]], left_on="match_id", right_on="id", how="left")

# Rank teams by their number of goals
df_team_rank = (df_goals.groupby("team")
                         .size()
                         .reset_index(name="goals")
                         .rename(columns={"team": "team_api_id"}))
df_team_rank = df_team_rank.merge(df_team[["team_api_id", "team_long_name"]], on="team_api_id",how="left")
df_team_rank = df_team_rank.sort_values("goals", ascending=False)
df_team_rank["team_rank"] = range(1,len(df_team_rank) + 1)

## Borussia Dortmund ID ##
dortmund = df_team[df_team["team_long_name"] == "Borussia Dortmund"]
if len(dortmund) == 0:
    raise ValueError("Borussia Dortmund was not found in Team.csv.")
dortmund_id = dortmund["team_api_id"].iloc[0]


## Generating Chart

In [4]:
''' Heat map / Scatter plot '''

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container([
    html.H2("Crossing Frequency", className="my-4"),
    # Drop down for team slection
    html.Label("Select Team Group"),
    dcc.Dropdown(
        id="team-group-dropdown",
        options=[{"label": "All Teams", "value": "all"},
                 {"label": "Top 100 Teams", "value": 100},
                 {"label": "Top 50 Teams", "value": 50},
                 {"label": "Top 20 Teams", "value": 20},
                 {"label": "Top 15 Teams", "value": 15},
                 {"label": "Top 10 Teams", "value": 10}],
        value="all",
        clearable=False,
        className="mb-4"),
    dcc.Graph(id="crossing-frequency-chart")])

@app.callback(Output("crossing-frequency-chart", "figure"),Input("team-group-dropdown", "value"))
# Updating plot according to what user selects in drop down menu
def update_crossing_chart(selected_top_n):
    if selected_top_n == "all":
        selected_teams = set(df_team_rank["team_api_id"])
    else:
        top_teams = df_team_rank[df_team_rank["team_rank"] <= selected_top_n]["team_api_id"]
        selected_teams = set(top_teams)
    selected_teams.add(dortmund_id)

    df_plot = df_crossing_agg[df_crossing_agg["team_api_id"].isin(selected_teams)].copy()
    df_plot["is_dortmund"] = (df_plot["team_long_name"] == "Borussia Dortmund")
    df_plot["x_numeric"] = (df_plot["time_label"].map(time_dict).astype(float))
    df_plot["x_jitter"] = (df_plot["x_numeric"] +np.random.uniform(-0.12, 0.12, size=len(df_plot)))

    # Two plots for "wins" and "draws and losses"
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Wins", "Draws and Losses"),shared_yaxes=True)
    result_groups = ["Wins", "Draws and Losses"]

    for col, result_group in enumerate(result_groups, start=1):
        df_result = df_plot[df_plot["result_group"] == result_group]
        df_other = df_result[df_result["is_dortmund"] == False]
        df_dortmund = df_result[df_result["is_dortmund"] == True]
        # Marker teams
        fig.add_trace(
            go.Scatter(
                x=df_other["x_jitter"],
                y=df_other["cross_shot_percentage"],
                mode="markers",
                marker=dict(symbol="circle", size=6, color="black", opacity=0.5),
                text=df_other["team_long_name"],
                customdata=df_other[["time_label", "crosses", "total_shots"]],
                hovertemplate=("<b>%{text}</b><br>"
                               "Time: %{customdata[0]}<br>"
                               "Crossing frequency: %{y:.1f}%<br>"
                               "Crosses: %{customdata[1]}<br>"
                               "Total shots: %{customdata[2]}"
                               "<extra></extra>"),
                name="Other teams",
                showlegend=(col == 1)),
            row=1,
            col=col)
        # Specialmarker for BVB
        fig.add_trace(
            go.Scatter(
                x=df_dortmund["x_jitter"],
                y=df_dortmund["cross_shot_percentage"],
                mode="markers",
                marker=dict(symbol="circle", size=10, color="orange", line=dict(color="black")),
                text=df_dortmund["team_long_name"],
                customdata=df_dortmund[["time_label", "crosses", "total_shots"]],
                hovertemplate=("<b>%{text}</b><br>"
                               "Time: %{customdata[0]}<br>"
                               "Crossing frequency: %{y:.1f}%<br>"
                               "Crosses: %{customdata[1]}<br>"
                               "Total shots: %{customdata[2]}"
                               "<extra></extra>"),
                name="Borussia Dortmund",
                showlegend=(col == 1)),
            row=1,
            col=col)
        # Heat map
        fig.add_hrect(y0=0,  y1=20, fillcolor="lightgreen", opacity=0.20, line_width=0, row=1, col=col)
        fig.add_hrect(y0=20, y1=40, fillcolor="lightyellow", opacity=0.25, line_width=0, row=1, col=col)
        fig.add_hrect(y0=40, y1=60, fillcolor="khaki", opacity=0.25, line_width=0, row=1, col=col)
        fig.add_hrect(y0=60, y1=80, fillcolor="lightsalmon", opacity=0.20, line_width=0, row=1, col=col)
        fig.add_hrect(y0=80, y1=100, fillcolor="lightcoral", opacity=0.20, line_width=0, row=1, col=col)
    # X-Axis
    fig.update_xaxes(
        type="linear",
        range=[0.5, 6.5],
        tickvals=[1, 2, 3, 4, 5, 6],
        ticktext=time_label,
        title_text="Portion of the Game [Min]")
    # Y-Axis
    fig.update_yaxes(
        title_text="Crossing Frequency (%)",
        range=[0, 100],
        tickvals=[20, 40, 60, 80, 100],
        ticktext=["≤20%", "21-40%", "41-60%", "61-80%", "81-100%"])
    # Title
    fig.update_layout(
        template="plotly_white",
        height=500,
        title="Crossing Frequency by Match Outcome",
        legend_title="Team")
    return fig

app.run(debug=True, port = 8070)
